# fMRI Analysis Pipeline Configuration

This notebook helps create and manage configuration files for the fMRI first-level analysis pipeline. It provides an interactive interface to:

1. Create configuration files for one or multiple subjects
2. Customize analysis parameters
3. Define contrasts of interest
4. Generate batch job scripts for HPC execution

## Setup

In [ ]:
import os
import json
import glob
import pandas as pd
import numpy as np
from pathlib import Path

# Import our configuration utilities
from configuration_utils import (
    create_config_from_bids,
    save_config,
    validate_configuration,
    load_events_and_add_contrasts
)

## Project Settings

Set up the basic parameters for your project:

In [ ]:
# Data paths
project_dir = "/data00/projects/geoscan_v2"
bids_root = os.path.join(project_dir, "data/bids_data")
output_dir = os.path.join(bids_root, "derivatives/nipype")
working_dir = os.path.join(bids_root, "working/nipype")

# Software paths
spm_path = "/data00/tools/spm12mega"
fsl_path = "/data00/tools/fsl"

# Analysis parameters
task = "image"  # Task name
model_name = "GS-condition"  # Model name
smoothing = 6.0  # FWHM for smoothing in mm
high_pass = 160  # High-pass filter cutoff in seconds
tr = 3.0  # Repetition time in seconds

# Default model parameters
serial_correlations = "FAST"  # Options: AR(1), FAST, None
global_intensity = "none"  # Options: none, scaling, grand mean scaling

## Find Available Subjects

In [ ]:
# Find all subjects with the task of interest
subject_dirs = glob.glob(os.path.join(bids_root, "derivatives/fmriprep/sub-*"))
available_subjects = []

for subject_dir in subject_dirs:
    subject_id = os.path.basename(subject_dir)
    func_files = glob.glob(
        os.path.join(subject_dir, "ses-*/func/*_task-" + task + "_*_space-MNI152NLin2009cAsym_desc-preproc_bold.nii*")
    )
    
    if func_files:
        # Extract session information
        sessions = set()
        for func_file in func_files:
            parts = os.path.basename(func_file).split("_")
            session = next((part for part in parts if part.startswith("ses-")), None)
            if session:
                sessions.add(session)
        
        available_subjects.append({
            "subject": subject_id.replace("sub-", ""),
            "sessions": list(sessions),
            "num_runs": len(func_files)
        })

# Display available subjects
pd.DataFrame(available_subjects)

## Select Subjects for Analysis

In [ ]:
# Select subjects to process
selected_subjects = ["GS028", "GS029", "GS030"]  # Modify this list as needed
selected_session = "t2"  # Change if needed

## Customize the Model

Define the contrast matrix for your task.

In [ ]:
# Define contrasts for the GeoScan Image task
geoscan_contrasts = [
    ["stimuli_rest", "T", ["stimuli_rest"], [1]],
    ["rating", "T", ["rating"], [1]],
    ["retail_smoke_register", "T", ["retail_smoke_register"], [1]],
    ["retail_nonsmoke_promo", "T", ["retail_nonsmoke_promo"], [1]],
    ["standard_nonsmoke", "T", ["standard_nonsmoke"], [1]],
    ["retail_smoke_promo", "T", ["retail_smoke_promo"], [1]],
    ["retail_nonsmoke_register", "T", ["retail_nonsmoke_register"], [1]],
    ["retail_smoke_branded", "T", ["retail_smoke_branded"], [1]],
    ["standard_smoke", "T", ["standard_smoke"], [1]],
    ["standardSmoke_v_standardNonsmoke", "T", ["standard_smoke", "standard_nonsmoke"], [1, -1]],
    ["retailSmokeRegister_v_retailNonsmokeRegister", "T", ["retail_smoke_register", "retail_nonsmoke_register"], [1, -1]],
    ["retailSmoke_v_retailNonsmoke", "T", 
        ["retail_smoke_register", "retail_smoke_branded", "retail_smoke_promo", "retail_nonsmoke_register", "retail_nonsmoke_promo"], 
        [0.33, 0.33, 0.34, -0.5, -0.5]
    ]
]

## Generate Configuration Files

In [ ]:
def create_custom_config(subject, session):
    """Create a custom configuration for a subject/session"""
    # Find functional files
    func_pattern = f"derivatives/fmriprep/sub-{subject}/ses-{session}/func/sub-{subject}_ses-{session}_task-{task}_run-*_space-MNI152NLin2009cAsym_desc-preproc_bold.nii*"
    func_files = glob.glob(os.path.join(bids_root, func_pattern))
    func_files.sort()
    
    if not func_files:
        print(f"No functional files found for sub-{subject} ses-{session}")
        return None
    
    # Make paths relative to project_dir
    func_files_rel = [os.path.relpath(f, project_dir) for f in func_files]
    
    # Find event files
    event_files = []
    for func_file in func_files:
        # Extract run number
        parts = os.path.basename(func_file).split("_")
        run_part = next((part for part in parts if part.startswith("run-")), None)
        
        if run_part:
            # Create model event path
            event_file = f"models/task-{task}_model-{model_name}/events/sub-{subject}_ses-{session}_task-{task}_{run_part}_events.tsv"
            event_files.append(event_file)
    
    # Find regressor files
    regressor_files = []
    for func_file in func_files:
        # Extract run number
        parts = os.path.basename(func_file).split("_")
        run_part = next((part for part in parts if part.startswith("run-")), None)
        
        if run_part:
            # Create regressor path
            regressor_file = f"data/bids_data/derivatives/outlier/regressors/sub-{subject}/sub-{subject}_ses-{session}_task-{task}_{run_part}_desc-motion_regressors.tsv"
            regressor_files.append(regressor_file)
    
    # Create configuration
    config = {
        "Description": [
            f"Model by condition",
            f"GS highpass {high_pass} R01 default pipeline",
            f"{smoothing}mm FWHM smoothing",
            "No global scaling",
            f"{serial_correlations} correlation",
            "x_trans, y_trans, z_trans, x_rot, y_rot, z_rot, csf, trash regressor (FD > 0.75 | GS > 3 SD)",
            "Geoscan Image Task"
        ],
        "Template": [
            "/data00/projects/geoscan_v2/scripts/L1/study-geoscanR01-GS.json",
            "/data00/projects/geoscan_v2/scripts/L1/task_images-GS.json"
        ],
        "Info": {
            "model": model_name,
            "task": task,
            "sub": subject
        },
        "Environment": {
            "spm_path": spm_path,
            "fsl_path": fsl_path,
            "data_path": project_dir,
            "output_path": output_dir,
            "working_path": working_dir
        },
        "IsotropicSmooth": {
            "fwhm": smoothing
        },
        "SpecifySPMModel": {
            "input_units": "secs",
            "output_units": "secs",
            "high_pass_filter_cutoff": high_pass,
            "time_repetition": tr,
            "functional_runs": func_files_rel,
            "event_files": event_files,
            "regressors": regressor_files,
            "regressor_names": ["trans_x", "trans_y", "trans_z", "rot_x", "rot_y", "rot_z", "csf", "trash"]
        },
        "Level1Design": {
            "bases": {"hrf": {"derivs": [0, 0]}},
            "timing_units": "secs",
            "global_intensity_normalization": global_intensity,
            "model_serial_correlations": serial_correlations,
            "interscan_interval": tr
        },
        "EstimateModel": {
            "estimation_method": {"Classical": 1},
            "write_residuals": False
        },
        "EstimateContrast": {
            "contrasts": geoscan_contrasts
        }
    }
    
    return config

In [ ]:
# Create configurations for all selected subjects
configs = {}
config_files = {}

for subject in selected_subjects:
    config = create_custom_config(subject, selected_session)
    
    if config:
        # Validate configuration
        errors = validate_configuration(config, check_files=False)
        if errors:
            print(f"Validation errors for sub-{subject}:")
            for error in errors:
                print(f"  - {error}")
        else:
            configs[subject] = config
            
            # Save configuration
            output_file = f"sub-{subject}.json"
            with open(output_file, 'w') as f:
                json.dump(config, f, indent=4)
            
            config_files[subject] = output_file
            print(f"Created configuration for sub-{subject}: {output_file}")

print(f"\nGenerated {len(configs)} configuration files")

## Generate SLURM Batch Script

In [ ]:
def create_slurm_script(config_files, output_file="run_l1analysis.sh"):
    """Create a SLURM batch script for processing multiple subjects"""
    
    script = f"""#!/bin/bash
#SBATCH --job-name=fmri_l1
#SBATCH --output=logs/fmri_l1_%A_%a.out
#SBATCH --error=logs/fmri_l1_%A_%a.err
#SBATCH --array=1-{len(config_files)}
#SBATCH --time=2:00:00
#SBATCH --mem=8G
#SBATCH --cpus-per-task=2

# Create logs directory if it doesn't exist
mkdir -p logs

# Define the config files to process
declare -a config_files=(
"""
    
    # Add each config file
    for config_file in config_files.values():
        script += f"    \"{config_file}\"\n"
    
    script += """)\n\n# Get the config file for this job array task\nCONFIG_FILE=${config_files[$SLURM_ARRAY_TASK_ID-1]}\n\necho "Processing $CONFIG_FILE"\n\n# Run the analysis\npython improved_pipeline.py $CONFIG_FILE\n\necho "Finished processing $CONFIG_FILE"\n"""
    
    with open(output_file, 'w') as f:
        f.write(script)
    
    # Make executable
    os.chmod(output_file, 0o755)
    
    return output_file

In [ ]:
# Create SLURM batch script
slurm_script = create_slurm_script(config_files)
print(f"Created SLURM batch script: {slurm_script}")

## Submit Batch Jobs

You can run the following in a terminal to submit the batch jobs:

In [ ]:
print("Run the following command in a terminal to submit the batch jobs:")
print(f"sbatch {slurm_script}")

## Test Single Subject Processing

You can process a single subject to test the pipeline:

In [ ]:
# Choose a subject to test
test_subject = selected_subjects[0]
test_config = config_files[test_subject]

print(f"To test processing for {test_subject}, run:")
print(f"python improved_pipeline.py {test_config}")

## Monitor Progress

In [ ]:
def check_outputs(subject, session):
    """Check if outputs exist for a subject"""
    output_base = os.path.join(output_dir, f"task-{task}_model-{model_name}", f"sub-{subject}")
    
    expected_outputs = {
        "SPM.mat": os.path.join(output_base, "spm", "SPM.mat"),
        "Beta images": glob.glob(os.path.join(output_base, "betas", "beta_*.nii")),
        "Contrast images": glob.glob(os.path.join(output_base, "con", "con_*.nii")),
        "T-stat images": glob.glob(os.path.join(output_base, "spmT", "spmT_*.nii"))
    }
    
    results = {}
    for output_type, output_path in expected_outputs.items():
        if isinstance(output_path, list):
            results[output_type] = len(output_path)
        else:
            results[output_type] = os.path.exists(output_path)
    
    return results

# Check outputs for processed subjects
for subject in selected_subjects:
    outputs = check_outputs(subject, selected_session)
    print(f"\nSub-{subject}:")
    for output_type, status in outputs.items():
        print(f"  {output_type}: {status}")

## Troubleshooting

In [ ]:
def check_log_files(subject):
    """Check log files for errors"""
    log_pattern = f"logs/fmri_l1_*_{selected_subjects.index(subject)+1}.out"
    log_files = glob.glob(log_pattern)
    
    if log_files:
        latest_log = max(log_files, key=os.path.getctime)
        print(f"Latest log for {subject}: {latest_log}")
        
        # Check for common errors
        with open(latest_log, 'r') as f:
            content = f.read()
            if "error" in content.lower():
                print("Errors found in log file!")
                # Print last 20 lines
                lines = content.split('\n')
                print('\n'.join(lines[-20:]))
    else:
        print(f"No log files found for {subject}")

# Check logs for a specific subject
# check_log_files(selected_subjects[0])

## Advanced Configuration

### Add Parametric Modulators

In [ ]:
# Example: Add reaction time as parametric modulator
from configuration_utils import add_parametric_modulators

# Define parametric modulators for conditions
pmod_mapping = {
    "rating": "reaction_time",  # Use reaction_time column as modulator for rating events
    "stimuli_rest": ["duration", "intensity"]  # Multiple modulators
}

# Add to existing config
config_with_pmod = add_parametric_modulators(configs[selected_subjects[0]], pmod_mapping)

### Custom Regressor Selection

In [ ]:
# Select specific regressors from confound files
custom_regressors = [
    "trans_x", "trans_y", "trans_z",
    "rot_x", "rot_y", "rot_z",
    "framewise_displacement",
    "a_comp_cor_00", "a_comp_cor_01", "a_comp_cor_02",
    "csf", "white_matter"
]

# Update config
configs[selected_subjects[0]]["SpecifySPMModel"]["regressor_names"] = custom_regressors

### Multiple Models

In [ ]:
# Create different model configurations
models = {
    "basic": {
        "name": "basic",
        "contrasts": [
            ["task_vs_rest", "T", ["stimuli_rest", "rating"], [0.5, 0.5]]
        ]
    },
    "detailed": {
        "name": "detailed", 
        "contrasts": geoscan_contrasts
    }
}

# Generate configs for each model
for model_key, model_info in models.items():
    for subject in selected_subjects:
        config = create_custom_config(subject, selected_session)
        config["Info"]["model"] = model_info["name"]
        config["EstimateContrast"]["contrasts"] = model_info["contrasts"]
        
        output_file = f"sub-{subject}_model-{model_key}.json"
        save_config(config, output_file)